Import libraries

In [1]:
import json
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from dotenv import load_dotenv
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine

sys.path.insert(0, "..")
from app.features import FEATURES, add_history

MIN_SEASON = 1961  # expansion era
MIN_TARGET_PA = 100
SHORT_SEASON = 2020  # 60 games, so counting stats aren't comparable
TRAIN_END = 2009
VALIDATION_END = 2014
TARGETS = ["wRAA", "BsR", "RLR"]
MODEL_DIR = Path("../app/model")

Connect to db

In [2]:
load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))
player = pd.read_sql("SELECT * FROM playerstat", engine)
league = pd.read_sql("SELECT * FROM leaguestat", engine).rename(columns={"Season": "season"})
combined = pd.merge(player, league, on="season")
print(len(player), "player seasons,", len(league), "league seasons")

106604 player seasons, 154 league seasons


Compute wRAA, BsR, RLR and offensive WAR

Follows the FanGraphs definitions, without UBR or fielding, which this data doesn't have. League games are estimated as IP / 18 so replacement level scales with shorter seasons and fewer teams.

In [3]:
runs_per_out = combined["RS"] / (combined["IP"] * 3)
league_games = combined["IP"] / 18

Sabermetric = pd.DataFrame()
Sabermetric["IDfg"] = combined["idfg"]
Sabermetric["Age"] = combined["age"]
Sabermetric["Season"] = combined["season"]
Sabermetric["PA"] = combined["pa"]
Sabermetric["wRAA"] = ((combined["woba"] - combined["league_wOBA"]) / 1.25) * combined["pa"]  # wOBA scale = 1.25

Sabermetric["wGDP"] = ((combined["GDP"] / combined["GDP_OPP"]) * combined["gdp_opp"] - combined["gdp"]) * runs_per_out
Sabermetric["runCS"] = -(2 * runs_per_out + 0.075)
Sabermetric["league_wSB"] = ((combined["SB"] * 0.2) + (combined["CS"] * Sabermetric["runCS"])) / (
    combined["1B_y"] + combined["BB"] + combined["HBP"] - combined["IBB"]
)
Sabermetric["wSB"] = ((combined["sb"] * 0.2) + (combined["cs"] * Sabermetric["runCS"])) - (
    Sabermetric["league_wSB"] * (combined["1B_x"] + combined["bb"] + combined["hbp"] - combined["ibb"])
)
Sabermetric["BsR"] = Sabermetric["wSB"] + Sabermetric["wGDP"]

Sabermetric["RPW"] = 9 * (combined["RS"] / combined["IP"]) * 1.5 + 3
Sabermetric["RLR"] = (0.235 * league_games * Sabermetric["RPW"] * combined["pa"]) / combined["PA"]
Sabermetric["oWAR"] = (Sabermetric["wRAA"] + Sabermetric["BsR"] + Sabermetric["RLR"]) / Sabermetric["RPW"]

Sabermetric = Sabermetric.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
print(len(Sabermetric), "rows")
known = [("ruthba01", 1921), ("bondsba01", 2004), ("judgeaa01", 2022), ("troutmi01", 2016)]
print(pd.concat(Sabermetric[(Sabermetric.IDfg == i) & (Sabermetric.Season == s)] for i, s in known)[
    ["IDfg", "Season", "PA", "wRAA", "BsR", "RLR", "RPW", "oWAR"]
].round(2).to_string(index=False))

88369 rows
     IDfg  Season    PA   wRAA  BsR   RLR   RPW  oWAR
 ruthba01    1921 693.0 121.70 0.23 21.50 10.32 13.90
bondsba01    2004 617.0 102.34 1.07 19.05 10.27 11.92
judgeaa01    2022 692.0  80.88 0.74 20.37  9.52 10.71
troutmi01    2016 681.0  54.63 3.81 20.40  9.78  8.06


Load Sabermetric onto DB

In [4]:
Sabermetric.to_sql(name="playersabermetric", con=engine, if_exists="replace", index=False)

369

Build next-season data

Each row predicts season N using only seasons N-1 to N-3, plus the player's age in N. Rows are limited to seasons from 1961 onward, so predictions answer: if this player gets at least 100 PA next season, what will they produce?

In [5]:
modern = Sabermetric[Sabermetric["Season"] >= MIN_SEASON]

data = add_history(modern[(modern["PA"] >= MIN_TARGET_PA) & (modern["Season"] != SHORT_SEASON)], modern)
train = data[data["Season"] <= TRAIN_END]
validation = data[(data["Season"] > TRAIN_END) & (data["Season"] <= VALIDATION_END)]
test = data[data["Season"] > VALIDATION_END]
print(f"train {len(train)} ({train.Season.min()}-{train.Season.max()}), "
      f"validation {len(validation)} ({validation.Season.min()}-{validation.Season.max()}), "
      f"test {len(test)} ({test.Season.min()}-{test.Season.max()})")

train 16667 (1962-2009), validation 2002 (2010-2014), test 3603 (2015-2024)


Define baselines and models

Repeating last season and a Marcel-style projection set the bar the models have to beat.

In [6]:
def baseline_repeat(rows, target):
    return rows[f"{target}_l1"].to_numpy()


def baseline_marcel(rows, target):
    rate = rows["RLR_rate_l1"] if target == "RLR" else rows[f"{target}_rate"]
    return (rate * rows["PA_expected"]).to_numpy()


def fit_sklearn(estimator):
    def fit(rows, target):
        model = estimator().fit(rows[FEATURES], rows[target])
        return lambda new: model.predict(new[FEATURES]), model
    return fit


def make_nn(n_features):
    return nn.Sequential(nn.Linear(n_features, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1))


def fit_nn(rows, target):
    torch.manual_seed(42)
    scaler_X, scaler_y = StandardScaler(), StandardScaler()
    X = torch.tensor(scaler_X.fit_transform(rows[FEATURES].values), dtype=torch.float32)
    y = torch.tensor(scaler_y.fit_transform(rows[[target]].values), dtype=torch.float32)

    model = make_nn(len(FEATURES))
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-3)
    loss_fn = nn.MSELoss()
    for _ in range(400):
        loss = loss_fn(model(X), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    model.eval()

    def predict(new):
        with torch.no_grad():
            scaled = model(torch.tensor(scaler_X.transform(new[FEATURES].values), dtype=torch.float32)).numpy()
        return scaler_y.inverse_transform(scaled).ravel()

    return predict, {"state_dict": model.state_dict(), "scaler_X": scaler_X, "scaler_y": scaler_y}


MODELS = {
    "Ridge": fit_sklearn(lambda: make_pipeline(StandardScaler(), Ridge(alpha=10.0))),
    "GradientBoosting": fit_sklearn(lambda: HistGradientBoostingRegressor(max_depth=4, learning_rate=0.05, max_iter=200, random_state=42)),
    "RandomForest": fit_sklearn(lambda: RandomForestRegressor(n_estimators=200, min_samples_leaf=10, n_jobs=-1, random_state=42)),
    "NeuralNetwork": fit_nn,
}

Pick the best model for each target on the validation seasons

In [7]:
validation_scores, best = {}, {}
for target in TARGETS:
    scores = {name: r2_score(validation[target], fit(train, target)[0](validation)) for name, fit in MODELS.items()}
    validation_scores[target] = scores
    best[target] = max(scores, key=scores.get)
print(pd.DataFrame(validation_scores).round(3).to_string())
print("\nSelected:", best)

                   wRAA    BsR    RLR
Ridge             0.338  0.332  0.447
GradientBoosting  0.346  0.343  0.463
RandomForest      0.333  0.341  0.459
NeuralNetwork     0.337  0.334  0.458

Selected: {'wRAA': 'GradientBoosting', 'BsR': 'GradientBoosting', 'RLR': 'GradientBoosting'}


Evaluate once on held-out seasons, trained on everything before them

In [8]:
fit_rows = pd.concat([train, validation])
test_predictions = {}
results = []


def record(method, target, truth, predicted):
    results.append({"method": method, "target": target, "R2": r2_score(truth, predicted), "MAE": mean_absolute_error(truth, predicted)})


for target in TARGETS:
    truth = test[target]
    record("Repeat last season", target, truth, baseline_repeat(test, target))
    record("Marcel-style", target, truth, baseline_marcel(test, target))
    for name, fit in MODELS.items():
        predicted = fit(fit_rows, target)[0](test)
        record(name, target, truth, predicted)
        if name == best[target]:
            test_predictions[target] = predicted
            record("Selected", target, truth, predicted)

marcel_war = sum(baseline_marcel(test, t) for t in TARGETS) / test["RPW_l1"].to_numpy()
selected_war = sum(test_predictions[t] for t in TARGETS) / test["RPW_l1"].to_numpy()
record("Repeat last season", "oWAR", test["oWAR"], test["oWAR_l1"])
record("Marcel-style", "oWAR", test["oWAR"], marcel_war)
record("Selected", "oWAR", test["oWAR"], selected_war)

summary = pd.DataFrame(results)
print("R2")
print(summary.pivot(index="method", columns="target", values="R2").round(3)[["oWAR", *TARGETS]].to_string())
print("\nMAE")
print(summary.pivot(index="method", columns="target", values="MAE").round(2)[["oWAR", *TARGETS]].to_string())

full_time = (test["PA"] >= 400).to_numpy()
print(f"\nFull-time hitters (PA >= 400, n={full_time.sum()}): oWAR R2 "
      f"selected={r2_score(test['oWAR'][full_time], selected_war[full_time]):.3f}, "
      f"Marcel-style={r2_score(test['oWAR'][full_time], marcel_war[full_time]):.3f}")

R2
target               oWAR   wRAA    BsR    RLR
method                                        
GradientBoosting      NaN  0.346  0.269  0.386
Marcel-style        0.352  0.310  0.244  0.302
NeuralNetwork         NaN  0.348  0.247  0.410
RandomForest          NaN  0.337  0.260  0.392
Repeat last season  0.139  0.082  0.006 -0.179
Ridge                 NaN  0.339  0.260  0.390
Selected            0.399  0.346  0.269  0.386

MAE
target              oWAR  wRAA   BsR   RLR
method                                    
GradientBoosting     NaN  8.33  0.82  3.39
Marcel-style        1.08  8.57  0.85  3.62
NeuralNetwork        NaN  8.33  0.84  3.32
RandomForest         NaN  8.43  0.83  3.39
Repeat last season  1.22  9.89  0.99  4.48
Ridge                NaN  8.42  0.83  3.39
Selected            1.03  8.33  0.82  3.39

Full-time hitters (PA >= 400, n=1777): oWAR R2 selected=0.237, Marcel-style=0.192


Refit the selected models on all seasons and save them

In [9]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
final_predictors = {}
for target in TARGETS:
    predict, artifact = MODELS[best[target]](data, target)
    final_predictors[target] = predict
    joblib.dump(artifact, MODEL_DIR / f"{target}.joblib", compress=3)

metadata = {
    "features": FEATURES,
    "models": best,
    "min_season": MIN_SEASON,
    "min_target_pa": MIN_TARGET_PA,
    "data_through": int(modern["Season"].max()),
    "test_seasons": sorted(int(s) for s in test["Season"].unique()),
    "test_results": results,
}
(MODEL_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))
print(sorted(p.name for p in MODEL_DIR.iterdir()))

['BsR.joblib', 'RLR.joblib', 'bsr_model.pth', 'metadata.json', 'rf_bsr_model.pkl', 'rf_rlr_model.pkl', 'rf_wraa.pkl', 'rf_wraa_model.pkl', 'rlr_model.pth', 'scaler_X_bsr.pkl', 'scaler_X_rlr.pkl', 'scaler_X_wraa.pkl', 'scaler_y_bsr.pkl', 'scaler_y_rlr.pkl', 'scaler_y_wraa.pkl', 'v2', 'wRAA.joblib', 'wraa_model.pth']


Preview next-season projections for active hitters

In [10]:
latest = int(modern["Season"].max())
active = modern[modern["Season"] == latest][["IDfg", "Age"]].copy()
active["Season"] = latest + 1
active["Age"] += 1
active = add_history(active, modern)
active = active[active["PA_l1"] + active["PA_l2"] + active["PA_l3"] >= MIN_TARGET_PA].copy()

for target in TARGETS:
    active[f"pred_{target}"] = final_predictors[target](active)
active["pred_oWAR"] = sum(active[f"pred_{t}"] for t in TARGETS) / active["RPW_l1"]
active = active.merge(player[["idfg", "name"]].drop_duplicates("idfg"), left_on="IDfg", right_on="idfg")

columns = ["name", "Age", "PA_l1", "oWAR_l1", "pred_wRAA", "pred_BsR", "pred_RLR", "pred_oWAR"]
print(len(active), "active hitters, projecting", latest + 1)
print(active.sort_values("pred_oWAR", ascending=False)[columns].head(10).round(2).to_string(index=False))
print(active[active["IDfg"] == "troutmi01"][columns].round(2).to_string(index=False))

549 active hitters, projecting 2025
             name  Age  PA_l1  oWAR_l1  pred_wRAA  pred_BsR  pred_RLR  pred_oWAR
    Shohei Ohtani 31.0  728.0    10.42      40.73      2.58     18.80       6.41
        Juan Soto 27.0  713.0     8.49      41.82     -0.97     19.98       6.28
      Aaron Judge 33.0  703.0    11.57      40.67     -0.70     18.78       6.07
   Yordan Alvarez 28.0  635.0     6.66      36.21     -0.71     17.82       5.51
       Bobby Witt 25.0  709.0     8.12      30.09      1.73     19.66       5.32
 Gunnar Henderson 24.0  719.0     6.71      30.30      1.43     19.26       5.27
Vladimir Guerrero 26.0  697.0     6.91      31.14     -1.12     19.41       5.10
  Freddie Freeman 36.0  638.0     4.79      26.42      0.16     18.82       4.69
       Matt Olson 31.0  685.0     3.57      25.78     -0.96     19.50       4.58
   Corbin Carroll 25.0  684.0     3.34      21.36      4.07     18.44       4.53
      name  Age  PA_l1  oWAR_l1  pred_wRAA  pred_BsR  pred_RLR  pred_oWAR